# 建筑 KPI 规则异常检测（`tool` 版）

本 notebook 与仓库根目录的 [`building_anomaly_detection.py`](../building_anomaly_detection.py) 配套，实现你提供的 **「并行症状层 + 空调根因层 + 相对/绝对双阈」** 矩阵（默认值见该模块内 `DetectionThresholds`）。

## 第一层：症状（每小时并行判定）

1. **水耗**：\((w-w_e)/\max(w_e,\varepsilon) > 20\%\) 且 \((w-w_e) > 1000\) L。  
2. **总电**：\((e-e_e)/\max(e_e,\varepsilon) > 15\%\) 且 \((e-e_e) > 0.10 \cdot \max(e_e-h_e,1)\)（基载比例门槛）。  
3. **空调电**：\((h-h_e)/\max(h_e,\varepsilon) > 20\%\) 且 \((h-h_e) \ge 20\) kWh。  
4. **供水偏离**：\(|s-s_e| > 1.5\) ℃。  
5. **回水偏离**：\(|r-r_e| > 2.0\) ℃。  

温度不作为孤立「唯一判死」条件，但与空调电组合形成根因向量（见第二层）。

## 第二层：空调根因（仅 `flag_hvac_energy` 为真时）

- **A**：制冷样工况 + 供水偏离 + 空调电异常。  
- **B**：制冷 + 空调电异常 + 小温差（\(r-s<3\)）等。  
- **C**：非营业时段 + 期望空调很低但实际很高 + 温差仍正常。  

## 数据流

1. 读入 `tool/output/kpi_with_anomalies.csv`（若无则回退根目录旧文件名）。  
2. `merge_expected_from_baseline` 合并 `*_exp`。  
3. `detect_building_kpi_anomalies(..., include_soft_symptoms=True)`：满幅度 + **软症状**（用于 `anomaly_phase=ramp`）。  
4. `evaluate_symptom_recall`：`union` 模式推荐与 ramp 一起评估。

若修改了 `building_anomaly_detection.py`，请 **重启内核** 或运行下方 `importlib.reload`。

In [1]:
import importlib
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def resolve_repo_root() -> Path:
    c = Path.cwd().resolve()
    for p in (c, c.parent):
        if (p / "building_anomaly_detection.py").is_file():
            return p
    raise FileNotFoundError("未找到 building_anomaly_detection.py")


ROOT = resolve_repo_root()
sys.path.insert(0, str(ROOT))

import building_anomaly_detection as _det

importlib.reload(_det)
from building_anomaly_detection import (
    merge_expected_from_baseline,
    detect_building_kpi_anomalies,
    evaluate_symptom_recall,
    attribute_anomaly_devices,
)

OUT_DIR = ROOT / "tool" / "output"
ANOM = OUT_DIR / "kpi_with_anomalies.csv"
BASE = OUT_DIR / "kpi_baseline_for_detection.csv"
LAB = OUT_DIR / "kpi_anomaly_labels.csv"

if not ANOM.is_file():
    ANOM = ROOT / "fwwb01_realistic_with_simulated_kpis_anomalies.csv"
    BASE = ROOT / "fwwb01_kpis_baseline_for_detection.csv"
    LAB = ROOT / "fwwb01_kpis_anomaly_labels.csv"

df = pd.read_csv(ANOM)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["building_id", "timestamp"]).reset_index(drop=True)

base = pd.read_csv(BASE)
base["timestamp"] = pd.to_datetime(base["timestamp"])

df = merge_expected_from_baseline(df, base)
df = detect_building_kpi_anomalies(df, include_soft_symptoms=True)

print(
    "满幅度症状: water",
    int(df["flag_water"].sum()),
    "elec",
    int(df["flag_electricity"].sum()),
    "hac",
    int(df["flag_hvac_energy"].sum()),
    "supply_dev",
    int(df["flag_supply_deviation"].sum()),
    "return_dev",
    int(df["flag_return_deviation"].sum()),
)
print("软症状(any):", int(df["flag_symptom_soft_any"].sum()), "并集(any):", int(df["flag_symptom_union_any"].sum()))

show_cols = [
    "timestamp",
    "building_id",
    "flag_water",
    "flag_electricity",
    "flag_hvac_energy",
    "flag_supply_deviation",
    "flag_return_deviation",
    "flag_symptom_soft_any",
    "flag_symptom_union_any",
    "hvac_diagnosis",
]
display(df[show_cols].head(15))

dev_csv = OUT_DIR / "kpi_detection_flags.csv"
if OUT_DIR.is_dir():
    export = df.copy()
    export["timestamp"] = export["timestamp"].dt.strftime("%Y-%m-%d %H:%M:%S")
    export.to_csv(dev_csv, index=False, encoding="utf-8-sig")
    print("exported:", dev_csv)

满幅度症状: water 178 elec 353 hac 1141 supply_dev 685 return_dev 863
软症状(any): 2837 并集(any): 2837


,timestamp,building_id,flag_water,flag_electricity,flag_hvac_energy,flag_supply_deviation,flag_return_deviation,flag_symptom_soft_any,flag_symptom_union_any,hvac_diagnosis
0,2016-01-01 00:00:00,Fox_education_Louie,False,False,False,False,False,False,False,
1,2016-01-01 01:00:00,Fox_education_Louie,False,False,False,False,False,False,False,
2,2016-01-01 02:00:00,Fox_education_Louie,False,False,False,False,False,False,False,
3,2016-01-01 03:00:00,Fox_education_Louie,False,False,False,False,False,False,False,
4,2016-01-01 04:00:00,Fox_education_Louie,False,False,False,False,False,False,False,
5,2016-01-01 05:00:00,Fox_education_Louie,False,False,False,False,False,False,False,
6,2016-01-01 06:00:00,Fox_education_Louie,False,False,False,False,False,False,False,
7,2016-01-01 07:00:00,Fox_education_Louie,False,False,False,False,False,False,False,
8,2016-01-01 08:00:00,Fox_education_Louie,False,False,False,False,False,False,False,
9,2016-01-01 09:00:00,Fox_education_Louie,False,False,False,False,False,False,False,


exported: E:\yp\data\tool\output\kpi_detection_flags.csv


In [2]:
import importlib
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def resolve_repo_root() -> Path:
    c = Path.cwd().resolve()
    for p in (c, c.parent):
        if (p / "building_anomaly_detection.py").is_file():
            return p
    raise FileNotFoundError("未找到 building_anomaly_detection.py")


ROOT = resolve_repo_root()
sys.path.insert(0, str(ROOT))
import building_anomaly_detection as _det2

importlib.reload(_det2)
from building_anomaly_detection import evaluate_symptom_recall, attribute_anomaly_devices

OUT_DIR = ROOT / "tool" / "output"
LAB = OUT_DIR / "kpi_anomaly_labels.csv"
if not LAB.is_file():
    LAB = ROOT / "fwwb01_kpis_anomaly_labels.csv"

lab = pd.read_csv(LAB)
lab["timestamp"] = pd.to_datetime(lab["timestamp"])


def _show(title: str, d: dict) -> None:
    print("===", title, "===")
    print("n_label_hours:", d["n_label_hours"], "symptom_recall:", round(d["symptom_recall"], 4))
    if "recall_by_anomaly_type" in d:
        print("by anomaly_type_id:")
        print(d["recall_by_anomaly_type"])
    if "recall_by_anomaly_phase" in d:
        print("by phase:")
        print(d["recall_by_anomaly_phase"])


_show("全部标签小时 | strict（满幅度）", evaluate_symptom_recall(df, lab, detection_mode="strict"))
_show("全部标签小时 | union（满幅度∪软）", evaluate_symptom_recall(df, lab, detection_mode="union"))
if "anomaly_phase" in lab.columns:
    _show("仅 full | strict", evaluate_symptom_recall(df, lab, label_phase="full", detection_mode="strict"))
    _show("仅 ramp | union（推荐）", evaluate_symptom_recall(df, lab, label_phase="ramp", detection_mode="union"))

dim_path = ROOT / "fwwb01_device_dimension.csv"
if dim_path.is_file():
    dim = pd.read_csv(dim_path)
    df_attr = attribute_anomaly_devices(df, dim, include_soft_flags=True)
    hit = df_attr["attributed_device_ids"] != "[]"
    print("设备归因行数（有 flag 或诊断）:", int(hit.sum()))
    display(df_attr.loc[hit, ["timestamp", "building_id", "attributed_device_types", "hvac_diagnosis"]].head(8))
else:
    print("未找到 fwwb01_device_dimension.csv，跳过设备归因")

=== 全部标签小时 | strict（满幅度） ===
n_label_hours: 2972 symptom_recall: 0.7746
by anomaly_type_id:
anomaly_type_id
1     0.919481
2     0.722222
3     0.733333
4     0.606987
5     0.669604
6     0.803030
7     0.750000
8     1.000000
9     0.850000
10    0.789809
11    0.862069
12    0.609375
13    0.804124
Name: caught, dtype: float64
by phase:
anomaly_phase
full    0.985141
ramp    0.600246
Name: caught, dtype: float64
=== 全部标签小时 | union（满幅度∪软） ===
n_label_hours: 2972 symptom_recall: 0.8496
by anomaly_type_id:
anomaly_type_id
1     0.948052
2     0.800000
3     0.828571
4     0.711790
5     0.769457
6     0.893939
7     0.854167
8     1.000000
9     0.930556
10    0.868365
11    0.908046
12    0.718750
13    0.845361
Name: caught, dtype: float64
by phase:
anomaly_phase
full    0.997028
ramp    0.727552
Name: caught, dtype: float64
=== 仅 full | strict ===
n_label_hours: 1346 symptom_recall: 0.9851
by anomaly_type_id:
anomaly_type_id
1     1.000000
2     1.000000
3     1.000000
4     1.00000

,timestamp,building_id,attributed_device_types,hvac_diagnosis
23,2016-01-01 23:00:00,Fox_education_Louie,"[""ac""]",
58,2016-01-03 10:00:00,Fox_education_Louie,"[""ac""]",
93,2016-01-04 21:00:00,Fox_education_Louie,"[""ac""]",
105,2016-01-05 09:00:00,Fox_education_Louie,"[""ac""]",
127,2016-01-06 07:00:00,Fox_education_Louie,"[""ac""]",
151,2016-01-07 07:00:00,Fox_education_Louie,"[""ac""]",
154,2016-01-07 10:00:00,Fox_education_Louie,"[""ac""]",
156,2016-01-07 12:00:00,Fox_education_Louie,"[""ac""]",
